# Applied Machine Learning, Clustering 4
## Building portfolios with clustering: from Markowitz to Hierarchical Risk Parity

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### Where we are coming from

In the previous three notebooks clustering grouped customers, stocks and households. Today it groups assets for a very
concrete task: **deciding how much money to put in each one**. It is the most direct application of clustering to
portfolio management, and to understand it we need to step back to how a portfolio is built and why the classic
method, Markowitz's, works on the blackboard and fails in practice.

The promise of the method we are going to build, **Hierarchical Risk Parity** (López de Prado, 2016), is modest and
honest: it does not promise more return, it promises **stable** portfolios, with no short positions, that do not need
to invert a matrix and that exploit a structure Markowitz ignores: assets resemble each other in groups. And we will
end with an uncomfortable question that is better asked before a student asks it: if splitting the money in equal
parts performs almost as well as every sophisticated method, why bother?

### What you will learn today

1. A portfolio refresher: return and risk of a combination of assets, why diversification works, and what Markowitz says.
2. The classic allocation rules, from the simplest to the most ambitious: 1/N, inverse variance, minimum variance, risk parity and maximum Sharpe.
3. Why the optimizer amplifies estimation errors, with the example of the maximum Sharpe portfolio blowing up.
4. HRP step by step: from correlations to distances, the dendrogram, quasi-diagonalization and recursive bisection.
5. An honest backtest with 49 industries, where 1/N almost wins, and why; and a second universe with bonds, gold and stocks where 1/N concentrates the risk and risk-based methods do matter.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import warnings, os, io, zipfile, urllib.request, time

from scipy.spatial.distance import squareform, pdist
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)


## 1. Refresher: what a portfolio is and what Markowitz says

A portfolio is a vector of weights $w = (w_1, \dots, w_N)$ that sum to 1: the fraction of the money in each asset. If
the assets have expected returns $\mu$ and covariance matrix $\Sigma$, the portfolio has

$$\text{expected return} = w^\top \mu, \qquad \text{variance} = w^\top \Sigma\, w = \sum_i \sum_j w_i w_j \sigma_{ij}.$$

The second formula is the one that explains diversification. The variance of the portfolio is not the average of the
variances: it depends on the **covariances** $\sigma_{ij}$ between each pair. If two assets do not move together,
their movements partly cancel out and the portfolio is less risky than either of them. With many weakly correlated
assets, the idiosyncratic risk of each one almost disappears and only what they have in common remains.

Markowitz (1952) turned that into an optimization problem: for each level of expected return, the minimum variance
portfolio; the set of those portfolios is the **efficient frontier**, and the best of all of them (maximum return per
unit of risk) is the **maximum Sharpe** portfolio. It is correct, it is elegant, and it has a problem that is not in
the mathematics but in the inputs: **we never know $\mu$ or $\Sigma$**. We estimate them from historical data, and
the optimizer treats every estimation error as information: if an asset happened by chance to have a high average
return in the sample, the optimizer loads the portfolio onto it. An average return is estimated very poorly (the
standard error of a daily average with one year of data is the size of the average itself), and that is why the
"optimal" portfolios change completely from one month to the next and perform worse than naive rules. This is the
result of DeMiguel, Garlappi and Uppal (2009): over dozens of datasets, no sophisticated rule consistently beat
splitting the money in equal parts.

### The allocation rules, from the simplest to the most ambitious

| rule | weights | what it needs to estimate |
|---|---|---|
| **1/N** (equal weights) | $w_i = 1/N$ | nothing |
| **inverse variance** | $w_i \propto 1/\sigma_i^2$ | only the variances |
| **minimum variance** | $w \propto \Sigma^{-1} \mathbf{1}$ | the whole matrix $\Sigma$, and it has to be inverted |
| **risk parity** | each asset contributes the same fraction of total variance | the whole matrix $\Sigma$ |
| **maximum Sharpe** | $w \propto \Sigma^{-1} \mu$ | $\Sigma$ and also $\mu$ |

The further down the table, the more there is to gain if the inputs were perfect and the more there is to lose when
they are not. Risk parity deserves a line: the risk contribution of asset $i$ is $w_i (\Sigma w)_i / (w^\top \Sigma w)$,
and the rule equalizes those contributions; with weakly correlated assets it looks like inverse variance, with
correlated assets it does not.


In [ ]:

def equiponderada(ret):
    return pd.Series(1 / ret.shape[1], index=ret.columns)

def inversa_varianza(ret):
    w = 1 / ret.var()
    return w / w.sum()

def minima_varianza(ret):                      # unconstrained: allows short positions
    cov = ret.cov().values
    w = np.linalg.pinv(cov) @ np.ones(len(cov))
    return pd.Series(w / w.sum(), index=ret.columns)

def minima_varianza_largo(ret):                # long positions only, by numerical optimization
    cov = ret.cov().values * 1e4; n = len(cov)
    r = minimize(lambda w: w @ cov @ w, np.ones(n) / n, jac=lambda w: 2 * cov @ w, bounds=[(0, 1)] * n,
                 constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(n)}], method="SLSQP", options={"maxiter": 1000, "ftol": 1e-12})
    return pd.Series(r.x, index=ret.columns)

def paridad_de_riesgo(ret):                    # equalizes risk contributions, long only
    cov = ret.cov().values * 1e4; n = len(cov)
    def desigualdad(w):
        rc = w * (cov @ w)
        return ((rc - rc.mean()) ** 2).sum()
    w0 = 1 / np.sqrt(np.diag(cov)); w0 = w0 / w0.sum()
    r = minimize(desigualdad, w0, bounds=[(1e-6, 1)] * n, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], method="SLSQP", options={"maxiter": 1000, "ftol": 1e-14})
    return pd.Series(r.x / r.x.sum(), index=ret.columns)

def maximo_sharpe(ret):                        # Markowitz's tangency portfolio, with sample averages
    cov, mu = ret.cov().values, ret.mean().values
    w = np.linalg.pinv(cov) @ mu
    return pd.Series(w / w.sum(), index=ret.columns)

def contribucion_al_riesgo(w, cov):
    w = w.values; v = w @ cov @ w
    return pd.Series(w * (cov @ w) / v, index=cov.index if hasattr(cov, "index") else None)


## 2. The data: 49 industries

**49 industry portfolios** from Kenneth French's data library: each series is the daily return of a portfolio holding
every US company in one industry, since 2000. It is a convenient universe for learning because it covers the whole
economy with few assets, with no gaps and no companies that disappear. Later on we use a second, very different
universe.


In [ ]:

URL_DATOS = ""     # <-- link to the course file (49_Industry_Portfolios_Daily.csv) if you have it; otherwise it is downloaded from Dartmouth
BASE_FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

def leer_french(texto, bloque=0):
    # French's files contain several blocks (value weighted, equal weighted...) separated by text headers
    lineas = texto.splitlines()
    inicios = [i for i, l in enumerate(lineas) if l.startswith(",") and i + 1 < len(lineas) and lineas[i + 1][:8].strip().isdigit()]
    ini = inicios[bloque]
    fin = next(j for j in range(ini + 1, len(lineas) + 1) if j == len(lineas) or not lineas[j][:8].strip().isdigit())
    d = pd.read_csv(io.StringIO("\n".join(lineas[ini:fin])), index_col=0)
    d.index = pd.to_datetime(d.index, format="%Y%m%d"); d.columns = [c.strip() for c in d.columns]
    return d.replace([-99.99, -999], np.nan) / 100

if URL_DATOS:
    texto = urllib.request.urlopen(normalizar_link(URL_DATOS)).read().decode("latin-1")
elif os.path.exists("49_Industry_Portfolios_Daily.csv"):
    texto = open("49_Industry_Portfolios_Daily.csv", encoding="latin-1").read()
else:
    peticion = urllib.request.Request(BASE_FRENCH + "49_Industry_Portfolios_daily_CSV.zip", headers={"User-Agent": "Mozilla/5.0"})
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(peticion, timeout=180).read())) as z:
        texto = z.read(z.namelist()[0]).decode("latin-1")
R = leer_french(texto).loc["2000":].dropna(axis=1).dropna()
vol_anual = R.std() * np.sqrt(252)
print(f"{len(R):,} days, {R.shape[1]} industries, from {R.index.min().date()} to {R.index.max().date()}")
print(f"Annual volatility: minimum {vol_anual.min():.0%} ({vol_anual.idxmin()}), median {vol_anual.median():.0%}, maximum {vol_anual.max():.0%} ({vol_anual.idxmax()})")
print("Average correlation between industries:", round(R.corr().values[np.triu_indices(R.shape[1], 1)].mean(), 2))


## 3. The estimation problem, made visible

Before any new method, let us look at the problem with the data itself. We estimate $\mu$ and $\Sigma$ with one year
(252 days) of returns, compute the weights of each rule, and repeat with the following year. If the inputs were well
estimated, the weights should look alike from one year to the next.


In [ ]:

REGLAS = {"1/N": equiponderada, "inverse variance": inversa_varianza, "minimum variance (unconstrained)": minima_varianza,
          "minimum variance (long only)": minima_varianza_largo, "risk parity": paridad_de_riesgo, "maximum Sharpe": maximo_sharpe}
a1, a2 = R.loc["2022"], R.loc["2023"]
filas = []
for nombre, f in REGLAS.items():
    w1, w2 = f(a1), f(a2)
    filas.append(dict(regla=nombre, peso_maximo_2022=w1.max(), peso_minimo_2022=w1.min(), cortos_2022=int((w1 < 0).sum()),
                      activos_efectivos_2022=1 / (w1 ** 2).sum(), rotacion_2022_a_2023=np.abs(w1 - w2).sum() / 2))
print("Weights estimated with 2022 and how much they change when re-estimated with 2023 (turnover = fraction of the portfolio that has to be traded):")
print(pd.DataFrame(filas).set_index("regla").round(3).to_string())


The maximum Sharpe row is Markowitz's problem in one line: weights of several times the capital, dozens of short
positions, and when re-estimating a year later the whole portfolio has to be turned over. Unconstrained minimum
variance also goes short and turns over a lot, though less, because it does not use $\mu$ (the worst estimated input)
but only $\Sigma$. The rules that only look at variances, or that look at nothing, barely move. There is one measure
we will use throughout the notebook: the **effective number of assets**, $1 / \sum_i w_i^2$, which equals $N$ when
the weights are equal and 1 when everything sits in a single asset.


## 4. Hierarchical Risk Parity, step by step

López de Prado's observation is that the optimizer treats the 49 assets as if they were all comparable with each
other: by inverting $\Sigma$ it lets any asset offset any other. But assets have **group structure**: banks resemble
each other, resemble insurers quite a bit, miners a little and gold not at all. HRP uses that structure in three steps:
it discovers the tree of similarities with hierarchical clustering, orders the assets according to the tree, and
splits the capital by walking the tree from top to bottom. It never inverts a matrix.

### 4.1 From correlations to distances

Clustering needs distances, and correlation is not one (it ranges from -1 to 1, and two identical assets would have
"distance" 1). Mantegna's (1999) transformation is a valid distance:

$$d_{ij} = \sqrt{\tfrac{1}{2}\,(1 - \rho_{ij})}$$

it equals 0 if $\rho = 1$, 1 if $\rho = -1$, and satisfies the triangle inequality. HRP adds one step: instead of
clustering by $d_{ij}$, it clusters by the Euclidean distance between the **columns** of the matrix $d$, that is, it
compares how each asset relates to the **whole** universe and not only to the other one.


In [ ]:

def distancia_mantegna(corr):
    return np.sqrt(np.clip(0.5 * (1 - corr), 0, None))

def distancia_de_distancias(D):
    return squareform(pdist(D, metric="euclidean"))

corr = R.corr()
D = distancia_mantegna(corr.values)
D2 = distancia_de_distancias(D)
iu = np.triu_indices_from(D, 1)
pares = pd.DataFrame({"a": np.array(R.columns)[iu[0]], "b": np.array(R.columns)[iu[1]], "distancia": D[iu], "correlacion": corr.values[iu]})
print("The five most similar industries:"); print(pares.nsmallest(5, "distancia").round(3).to_string(index=False))
print(); print("The five most different:"); print(pares.nlargest(5, "distancia").round(3).to_string(index=False))


### 4.2 The dendrogram

With the distance matrix we build a **hierarchical tree**: each industry starts on its own, the two closest ones are
merged, and this is repeated until a single group remains. The height of each merge is how different the merged
groups were. We need to decide how the distance between two already formed groups is measured (the **linkage
criterion**): `single` uses the closest pair, `complete` the farthest, `average` the mean and `ward` the one that
increases variance the least. The original HRP uses `single`; exercise 2 compares them.


In [ ]:

Z_single = linkage(squareform(D2, checks=False), method="single")
fig = ff.create_dendrogram(D2, labels=list(R.columns), distfun=lambda x: squareform(D2, checks=False), linkagefun=lambda x: Z_single, color_threshold=0.7 * Z_single[:, 2].max())
fig.update_layout(title="Dendrogram of the 49 industries (single linkage, distance of distances)", height=480, width=1000, yaxis_title="distance")
fig.update_xaxes(tickangle=90)
fig.show()

SECTOR = {"Banks": "financials", "Insur": "financials", "Fin": "financials", "RlEst": "financials", "Hardw": "technology", "Softw": "technology", "Chips": "technology", "LabEq": "technology",
          "Oil": "energy and materials", "Coal": "energy and materials", "Mines": "energy and materials", "Gold": "energy and materials", "Chems": "energy and materials", "Steel": "energy and materials",
          "Food": "consumer staples", "Soda": "consumer staples", "Beer": "consumer staples", "Smoke": "consumer staples", "Hshld": "consumer staples", "Util": "consumer staples",
          "Drugs": "health care", "MedEq": "health care", "Hlth": "health care", "Rtail": "consumer discretionary", "Meals": "consumer discretionary", "Clths": "consumer discretionary", "Toys": "consumer discretionary", "Fun": "consumer discretionary", "Autos": "consumer discretionary"}
Z_ward = linkage(squareform(D2, checks=False), method="ward")
grupos = fcluster(Z_ward, 6, criterion="maxclust")
for g in range(1, 7):
    miembros = [c for c, k in zip(R.columns, grupos) if k == g]
    print(f"Group {g} ({len(miembros):2d}): " + ", ".join(miembros))
print()
print("Correlation of gold with the other industries: average", round(corr["Gold"].drop("Gold").mean(), 2), "(the overall average is", round(corr.values[iu].mean(), 2), ")")


The algorithm never saw the word "bank" or "technology", and even so the groups are recognizable: consumer staples
and utilities together, natural resources together, the technology industries together. Two details are worth
noting. **Gold stands alone**: its correlation with the rest is the lowest in the matrix, and that makes it the only
genuine diversifier in the set. And there is one big group mixing banks, construction, machinery, transport and
retail: that is not a failure of the algorithm, it is that those industries share almost all their risk with the
market and, from a portfolio's point of view, they are **a single cyclical bet** even though the declared sectors are
different.

### 4.3 Quasi-diagonalization: ordering in order to see

A correlation matrix of 49 assets in alphabetical order shows nothing. Ordered according to the dendrogram, so that
similar assets sit next to each other, the blocks jump out. That reordering is called quasi-diagonalization and it is
the second step of HRP: the information ends up concentrated near the diagonal.


In [ ]:

def orden_cuasi_diagonal(Z):
    # walks the tree and returns the leaves in the order they appear (López de Prado, 2016)
    Z = Z.astype(int)
    orden = pd.Series([Z[-1, 0], Z[-1, 1]]); n_items = Z[-1, 3]
    while orden.max() >= n_items:
        orden.index = range(0, orden.shape[0] * 2, 2)
        df = orden[orden >= n_items]
        i, j = df.index, df.values - n_items
        orden[i] = Z[j, 0]
        orden = pd.concat([orden, pd.Series(Z[j, 1], index=i + 1)]).sort_index()
        orden.index = range(orden.shape[0])
    return orden.tolist()

orden = orden_cuasi_diagonal(Z_single)
etiquetas_ord = [R.columns[i] for i in orden]
fig = go.Figure()
fig.add_trace(go.Heatmap(z=corr.values, x=list(R.columns), y=list(R.columns), zmin=0, zmax=1, colorscale="RdYlBu_r", showscale=False, xaxis="x", yaxis="y"))
fig.add_trace(go.Heatmap(z=corr.iloc[orden, orden].values, x=etiquetas_ord, y=etiquetas_ord, zmin=0, zmax=1, colorscale="RdYlBu_r", xaxis="x2", yaxis="y2"))
fig.update_layout(title="Correlations in alphabetical order (left) and quasi-diagonalized by the dendrogram (right)", height=520, width=1000,
                  xaxis=dict(domain=[0, 0.47], showticklabels=False), yaxis=dict(showticklabels=False, autorange="reversed"), xaxis2=dict(domain=[0.53, 1], showticklabels=False), yaxis2=dict(anchor="x2", showticklabels=False, autorange="reversed"))
fig.show()
print("Order produced by the tree (first 20):", ", ".join(etiquetas_ord[:20]))


### 4.4 Recursive bisection

The third step splits the capital by walking the order of the tree:

1. All assets in a single group, with weight 1.
2. Split the group into two halves according to the quasi-diagonal order.
3. Compute the variance of each half as if, inside it, the weights were inverse to the variance.
4. Split the group's weight between the two halves in **inverse** proportion to their variance: the riskier half receives less.
5. Repeat on each half until reaching individual assets.

Since it only multiplies by factors between 0 and 1, it **never produces a negative weight**, and since it splits
first between large blocks, no single asset can take over. And it never inverts $\Sigma$: every decision is a
comparison between two blocks, so the method can be computed even with more assets than observations, where
Markowitz does not even exist.


In [ ]:

def varianza_de_grupo(cov, items):
    sub = cov.loc[items, items]
    w = 1 / np.diag(sub.values); w = w / w.sum()
    return float(w @ sub.values @ w)

def biseccion_recursiva(cov, orden_items):
    w = pd.Series(1.0, index=orden_items)
    grupos = [orden_items]
    while grupos:
        grupos = [g[a:b] for g in grupos for a, b in ((0, len(g) // 2), (len(g) // 2, len(g))) if len(g) > 1]
        for i in range(0, len(grupos), 2):
            izq, der = grupos[i], grupos[i + 1]
            v_izq, v_der = varianza_de_grupo(cov, izq), varianza_de_grupo(cov, der)
            alfa = 1 - v_izq / (v_izq + v_der)
            w[izq] *= alfa; w[der] *= 1 - alfa
    return w

def hrp(ret, criterio="single"):
    cov, c = ret.cov(), ret.corr()
    d2 = distancia_de_distancias(distancia_mantegna(c.values))
    Z = linkage(squareform(d2, checks=False), method=criterio)
    orden = [c.columns[i] for i in orden_cuasi_diagonal(Z)]
    return biseccion_recursiva(cov, orden).reindex(ret.columns)

REGLAS["HRP"] = hrp
w_hrp = hrp(R)
print(f"HRP weights over the full sample: they sum to {w_hrp.sum():.3f}, minimum {w_hrp.min():.3%}, maximum {w_hrp.max():.3%}, effective assets {1 / (w_hrp ** 2).sum():.1f} out of 49")
pesos = pd.DataFrame({n: f(R) for n, f in REGLAS.items() if n != "maximum Sharpe"}).loc[etiquetas_ord]
fig = px.bar(pesos.reset_index(names="industria").melt(id_vars="industria", var_name="regla", value_name="peso"), x="industria", y="peso", color="regla", barmode="group",
             title="Weights of each rule over the full sample (industries in dendrogram order)")
fig.update_layout(height=440, width=1000)
fig.show()


## 5. The honest test: out of sample, with a rolling window

The standard protocol: on each decision date, $\Sigma$ (and $\mu$ for maximum Sharpe) is estimated with the last
$T = 252$ days, each rule produces its weights, and those weights are held for 21 trading days; then we move forward
and re-estimate. No decision uses data after its date. We measure return, volatility, Sharpe, maximum drawdown and
**turnover** (what fraction of the portfolio has to be traded at each re-estimation, which is a real cost).


In [ ]:

def backtest(R, reglas, T=252, paso=21):
    acum = {m: [] for m in reglas}; rot = {m: [] for m in reglas}; previo = {m: None for m in reglas}; fechas = []
    V = R.values
    for t in range(T, len(R) - paso, paso):
        ventana, futuro = R.iloc[t - T:t], V[t:t + paso]
        for m, f in reglas.items():
            w = f(ventana)
            acum[m].append(futuro @ w.values)
            if previo[m] is not None:
                rot[m].append(np.abs(w.values - previo[m]).sum() / 2)
            previo[m] = w.values
        fechas.append(R.index[t:t + paso])
    carteras = pd.DataFrame({m: np.concatenate(v) for m, v in acum.items()}, index=np.concatenate(fechas))
    resumen = pd.DataFrame({m: dict(retorno=carteras[m].mean() * 252, volatilidad=carteras[m].std() * np.sqrt(252), sharpe=carteras[m].mean() / carteras[m].std() * np.sqrt(252),
                                    caida_maxima=((1 + carteras[m]).cumprod() / (1 + carteras[m]).cumprod().cummax() - 1).min(), rotacion_mensual=np.mean(rot[m])) for m in reglas}).T
    return resumen, carteras

t0 = time.time()
resumen49, carteras49 = backtest(R, REGLAS)
print(f"49 industries, 2001 to {R.index.max().year}, 252-day window, monthly re-estimation   [{time.time() - t0:.0f} s]")
print(resumen49.round(3).to_string())


In [ ]:

acum = (1 + carteras49.drop(columns="maximum Sharpe")).cumprod()
fig = go.Figure()
for c in acum.columns:
    fig.add_scatter(x=acum.index, y=acum[c], name=c, line=dict(width=3 if c in ("1/N", "HRP") else 1.3))
fig.update_layout(title="Out-of-sample cumulative capital, 49 industries (maximum Sharpe is left out of the chart: it blows up)", height=460, width=900, yaxis_type="log", yaxis_title="capital (log scale)")
fig.show()


### So why bother?

Read the table without prejudice. **Maximum Sharpe is a disaster** (volatility of hundreds of percent and a turnover
of dozens of times the portfolio per month): that is what happens when the optimizer feeds on sample averages.
**Everything else performs about the same**, and 1/N, which estimates nothing, has a Sharpe comparable to any of
them. The risk-based methods lower the volatility (unconstrained minimum variance the most, at the cost of shorts and
of turning over two thirds of the portfolio every month; HRP lowers it with five times less turnover) and the maximum
drawdown, but the improvement in Sharpe ranges from hundredths to a tenth at most (the best is long-only minimum
variance, 0.78 against 0.62), and exercise 1 shows how much of that survives costs.

There are two reasons, and both matter for knowing when to use each thing.

**First: in this universe the assets are alike.** The 49 industries have volatilities between 16% and 50%, with half
of them between 21% and 29%, and an average correlation of 0.56. When all assets have similar risk and move together,
splitting in equal parts is already almost risk parity, and there is not much that a $\Sigma$-based method can
improve. DeMiguel and his coauthors conclude that for an estimated rule to beat 1/N you need estimation windows of
decades, because the theoretical gain is small and estimation error eats it.

**Second: the advantage of the risk-based methods is not in the Sharpe but in the profile.** Less volatility, less
drawdown, no shorts, and in the case of HRP little turnover. For a real mandate with transaction costs, position
limits and a committee that asks about the maximum drawdown, that profile is worth more than a few hundredths of
Sharpe.

What it takes for 1/N to stop being enough is a universe where the assets do **not** resemble each other. That is
what comes next.


## 6. Where 1/N fails: a universe of different assets

Fourteen ETFs covering stocks (SPY, QQQ, IWM, EFA, EEM), government bonds (IEF, TLT), corporate bonds (LQD, HYG),
inflation (TIP), gold, silver, commodities and real estate (GLD, SLV, DBC, VNQ), daily since 2007. Volatilities range
from 6% (TIP) to 31% (SLV): here 1/N puts one fourteenth of the money in each one, but the **risk** is put in by the
most volatile ones.


In [ ]:

URL_ETF = "https://raw.githubusercontent.com/daviddiazsolis/clustering_finance_playground/main/notebooks/data/etf_precios.csv"
try:
    precios_etf = pd.read_csv(URL_ETF, index_col=0, parse_dates=True)
except Exception:
    import yfinance as yf
    precios_etf = yf.download(["SPY", "QQQ", "IWM", "EFA", "EEM", "TLT", "IEF", "LQD", "HYG", "GLD", "SLV", "DBC", "VNQ", "TIP"], start="2007-01-01", end="2025-12-31", auto_adjust=True, progress=False)["Close"].dropna()
precios_etf = precios_etf.drop(columns=[c for c in ["SHY"] if c in precios_etf.columns])
E = precios_etf.pct_change().dropna()
print(f"{len(E):,} days, {E.shape[1]} ETFs, from {E.index.min().date()} to {E.index.max().date()}")
print("Annual volatility:", (E.std() * np.sqrt(252)).sort_values().round(2).to_dict())

cov_e = E.cov()
contrib = pd.DataFrame({n: contribucion_al_riesgo(f(E), cov_e) for n, f in REGLAS.items() if n in ("1/N", "risk parity", "HRP")})
ACCIONES = ["SPY", "QQQ", "IWM", "EFA", "EEM"]
print()
print("Fraction of total risk that comes from the five stock ETFs (over the full sample):")
print(contrib.loc[ACCIONES].sum().round(2).to_string())
fig = px.bar(contrib.reset_index(names="ETF").melt(id_vars="ETF", var_name="regla", value_name="contribución al riesgo"), x="ETF", y="contribución al riesgo", color="regla", barmode="group",
             title="Who contributes the risk in each portfolio: 1/N splits the money, not the risk")
fig.update_layout(height=420, width=900, yaxis_tickformat=".0%")
fig.show()


In [ ]:

t0 = time.time()
resumen_etf, carteras_etf = backtest(E, REGLAS)
print(f"14 ETFs, 2008 to {E.index.max().year}, 252-day window, monthly re-estimation   [{time.time() - t0:.0f} s]")
print(resumen_etf.round(3).to_string())
acum = (1 + carteras_etf.drop(columns="maximum Sharpe")).cumprod()
fig = go.Figure()
for c in acum.columns:
    fig.add_scatter(x=acum.index, y=acum[c], name=c, line=dict(width=3 if c in ("1/N", "HRP") else 1.3))
fig.update_layout(title="Out-of-sample cumulative capital, 14 ETFs", height=460, width=900, yaxis_title="capital")
fig.show()


Here the story changes. Under 1/N, five stock ETFs contribute close to 60% of the risk of a fourteen-asset portfolio:
the "diversification" is nominal. The risk-based rules cut volatility and maximum drawdown to half or less, with a
similar Sharpe (better for risk parity and inverse variance, somewhat lower for HRP), and HRP does it with all
weights positive and without inverting any matrix. The price is a lower absolute return (the portfolio ends up loaded
with bonds); whoever wants the same return as 1/N with less risk would have to lever the risk portfolio, which is
exactly what risk parity funds do.

The practical rule that comes out of the two universes: **1/N is the mandatory starting point** and the benchmark
every rule has to beat; the risk-based methods (inverse variance, risk parity, HRP) pay off when the assets have very
different risks or when there are many assets relative to the data; and the methods that use estimated expected
returns never pay off with sample averages. HRP, in particular, is the one that behaves best when $N$ is large and
$T$ small, and exercise 3 shows it.

## 7. What to take away

1. The risk of a portfolio depends on the covariances, not the variances: diversifying means combining assets that do not move together.
2. Markowitz is correct with true inputs and fragile with estimated inputs; expected returns are the worst estimated, and the sample maximum Sharpe portfolio is unusable.
3. HRP: Mantegna distance, dendrogram, quasi-diagonalization, recursive bisection. No matrix inversion, no shorts, stable.
4. With similar assets, 1/N performs almost like everything else; with different assets, 1/N concentrates the risk in the volatile ones and the risk-based methods cut volatility in half.
5. The advantage of HRP is measured in volatility, maximum drawdown and turnover, not in Sharpe; and in the fact that it works when $N$ gets close to $T$.


## Exercises

1. **Transaction costs.** Subtract from the monthly return of each rule a cost proportional to its turnover: 10, 30
   and 50 basis points per unit turned over. Recompute the net Sharpe of each rule in both universes. From what cost
   onward does unconstrained minimum variance stop being the one with the lowest volatility per unit of cost, and what
   happens to HRP?
2. **The linkage criterion.** Repeat the 49-industry backtest with HRP using `single`, `complete`, `average` and
   `ward`, and compare volatility, Sharpe and turnover. Does it change as much as the choice of rule?
3. **When N gets close to T.** Repeat the 49-industry backtest with windows of 60, 120, 252, 504 and 1,000 days, only
   for 1/N, unconstrained minimum variance and HRP, and plot the out-of-sample volatility against $q = N/T$. What
   happens to minimum variance when $q$ gets close to 1, and what happens to HRP?


## Solutions


In [ ]:

# Exercise 1: costs proportional to turnover
def sharpe_neto(resumen, costo_pb):
    neto = resumen["retorno"] - resumen["rotacion_mensual"] * 12 * costo_pb / 1e4
    return neto / resumen["volatilidad"]
filas = {}
for nombre, res in [("49 industries", resumen49), ("14 ETFs", resumen_etf)]:
    for pb in [0, 10, 30, 50]:
        filas[(nombre, f"{pb} bp")] = sharpe_neto(res, pb)
print("Sharpe net of costs (maximum Sharpe is omitted):")
print(pd.DataFrame(filas).drop(index="maximum Sharpe").round(3).to_string())


With costs, unconstrained minimum variance loses its edge fast (it turns over two thirds of the portfolio every
month), the rules that barely turn over (1/N, inverse variance, risk parity) end up on top, and HRP, which turns over
somewhat more, loses a few hundredths. In the ETF universe risk parity and inverse variance stay above 1/N at any
reasonable cost; HRP ends up slightly below in Sharpe, but with half the volatility and half the maximum drawdown,
which is what a mandate with a risk limit looks at first.


In [ ]:

# Exercise 2: linkage criteria
t0 = time.time()
reglas_enlace = {f"HRP {c}": (lambda ret, c=c: hrp(ret, criterio=c)) for c in ["single", "complete", "average", "ward"]}
res_enlace, _ = backtest(R, reglas_enlace)
print(f"[{time.time() - t0:.0f} s]")
print(res_enlace.round(3).to_string())


The four criteria give very similar results: the linkage changes the shape of the tree, but recursive bisection
splits by variances and ends up in similar portfolios. The decision that matters is whether to use HRP or not, not
how to link.


In [ ]:

# Exercise 3: estimation window and q = N/T
t0 = time.time()
filas = []
for T in [60, 120, 252, 504, 1000]:
    res, _ = backtest(R, {k: REGLAS[k] for k in ["1/N", "minimum variance (unconstrained)", "HRP"]}, T=T)
    for m in res.index:
        filas.append(dict(T=T, q=R.shape[1] / T, regla=m, volatilidad=res.loc[m, "volatilidad"], rotacion=res.loc[m, "rotacion_mensual"]))
sens = pd.DataFrame(filas)
print(f"[{time.time() - t0:.0f} s]")
print(sens.pivot(index="T", columns="regla", values="volatilidad").round(3).to_string())
fig = px.line(sens, x="q", y="volatilidad", color="regla", markers=True, log_x=True, title="Out-of-sample volatility by q = N/T (windows from 1,000 to 60 days)")
fig.update_layout(height=400, width=760)
fig.show()


With short windows ($q$ close to 1) minimum variance goes out of control: the matrix it inverts is dominated by sample
noise and the inversion amplifies precisely the worst estimated directions. HRP stays stable over the whole range: it
is not the lowest-volatility rule when there is plenty of data, but it is never the worst, and that robustness is its
main argument.
